In [2]:
from sklearn.datasets import fetch_20newsgroups
import pandas as pd
def twenty_newsgroup_to_csv():
    newsgroups_train = fetch_20newsgroups(subset='train', remove=('headers', 'footers', 'quotes'))
    df = pd.DataFrame([newsgroups_train.data, newsgroups_train.target.tolist()]).T
    df.columns = ['text', 'target']
    targets = pd.DataFrame( newsgroups_train.target_names, columns=['title'])
    out = pd.merge(df, targets, left_on='target', right_index=True)
    out.to_csv('20_newsgroup.csv', index=False)
    
twenty_newsgroup_to_csv()

In [3]:
# from openai.embeddings_utils import get_embeddings
import openai, os, tiktoken, backoff

openai.api_key = os.environ.get("OPENAI_API_KEY")
embedding_model = "text-embedding-ada-002"
embedding_encoding = "cl100k_base"  # this the encoding for text-embedding-ada-002
batch_size = 2000
max_tokens = 8000  # the maximum for text-embedding-ada-002 is 8191
df = pd.read_csv('20_newsgroup.csv')
print("Number of rows before null filtering:", len(df))
df = df[df['text'].isnull() == False]
encoding = tiktoken.get_encoding(embedding_encoding)
df["n_tokens"] = df.text.apply(lambda x: len(encoding.encode(x)))
print("Number of rows before token number filtering:", len(df))
df = df[df.n_tokens <= max_tokens]
print("Number of rows data used:", len(df))


Number of rows before null filtering: 11314
Number of rows before token number filtering: 11096
Number of rows data used: 11044


In [5]:
def get_embeddings(list_of_text, model):
    response = client.embeddings.create(input=list_of_text, model=model)
    return [item.embedding for item in response.data]
    
@backoff.on_exception(backoff.expo, openai.error.RateLimitError)
def get_embeddings_with_backoff(prompts, model):
    embeddings = []
    for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size]
        embeddings += get_embeddings(list_of_text=batch, model=model)
    return embeddings
prompts = df.text.tolist()
prompt_batches = [prompts[i:i+batch_size] for i in range(0, len(prompts), batch_size)]
embeddings = []
for batch in prompt_batches:
    batch_embeddings = get_embeddings_with_backoff(prompts=batch, model=embedding_model)
    embeddings += batch_embeddings
df["embedding"] = embeddings
df.to_parquet("data/20_newsgroup_with_embedding.parquet", index=False)

NameError: name 'client' is not defined

In [4]:
import numpy as np
from sklearn.cluster import KMeans
embedding_df = pd.read_parquet("data/20_newsgroup_with_embedding.parquet")
matrix = np.vstack(embedding_df.embedding.values)
num_of_clusters = 20
kmeans = KMeans(n_clusters=num_of_clusters, init="k-means++", n_init=10, random_state=42)
kmeans.fit(matrix)
labels = kmeans.labels_
embedding_df["cluster"] = labels

In [6]:
embedding_df.head()

,text,target,title,n_tokens,embedding,cluster
0,I was wondering if anyone out there could enli...,7,rec.autos,121,"[-0.0391300804913044, 0.013502633199095726, -0...",0
1,\nIt depends on your priorities. A lot of peo...,7,rec.autos,108,"[-0.0011249205563217402, -0.00376517535187304,...",0
2,an excellent automatic can be found in the sub...,7,rec.autos,476,"[-0.018259447067975998, -0.008410007692873478,...",0
3,: Ford and his automobile. I need information...,7,rec.autos,86,"[-0.012589422054588795, 0.006539034191519022, ...",0
4,\nYo! Watch the attributions--I didn't say tha...,7,rec.autos,130,"[-0.0006192282889969647, -0.011226896196603775...",13


In [7]:
# 统计每个cluster的数量
new_df = embedding_df.groupby('cluster')['cluster'].count().reset_index(name='count')

# 统计这个cluster里最多的分类的数量
title_count = embedding_df.groupby(['cluster', 'title']).size().reset_index(name='title_count')
first_titles = title_count.groupby('cluster').apply(lambda x: x.nlargest(1, columns=['title_count']))
first_titles = first_titles.reset_index(drop=True)
new_df = pd.merge(new_df, first_titles[['cluster', 'title', 'title_count']], on='cluster', how='left')
new_df = new_df.rename(columns={'title': 'rank1', 'title_count': 'rank1_count'})

# 统计这个cluster里第二多的分类的数量
second_titles = title_count[~title_count['title'].isin(first_titles['title'])]
second_titles = second_titles.groupby('cluster').apply(lambda x: x.nlargest(1, columns=['title_count']))
second_titles = second_titles.reset_index(drop=True)
new_df = pd.merge(new_df, second_titles[['cluster', 'title', 'title_count']], on='cluster', how='left')
new_df = new_df.rename(columns={'title': 'rank2', 'title_count': 'rank2_count'})
new_df.fillna(0, inplace=True)
new_df['per_1'] = (new_df['rank1_count'] / new_df['count']).map(lambda x: '{:.2%}'.format(x))
new_df['per_1_2'] = ((new_df['rank1_count'] + new_df['rank2_count'])/ new_df['count']).map(lambda x: '{:.2%}'.format(x))

# 将缺失值替换为 0
# 输出结果
display(new_df)

/var/folders/r6/m94pzwms76lc737dhxsx_7wc0000gn/T/ipykernel_16350/997272759.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  first_titles = title_count.groupby('cluster').apply(lambda x: x.nlargest(1, columns=['title_count']))
/var/folders/r6/m94pzwms76lc737dhxsx_7wc0000gn/T/ipykernel_16350/997272759.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  second_titles = second_titles.groupby('cluster').appl

,cluster,count,rank1,rank1_count,rank2,rank2_count,per_1,per_1_2
0,0,522,rec.autos,432,comp.sys.mac.hardware,6.0,82.76%,83.91%
1,1,391,comp.sys.ibm.pc.hardware,101,comp.sys.mac.hardware,85.0,25.83%,47.57%
2,2,1060,talk.politics.misc,129,talk.religion.misc,60.0,12.17%,17.83%
3,3,381,rec.motorcycles,364,comp.sys.mac.hardware,1.0,95.54%,95.80%
4,4,783,comp.sys.ibm.pc.hardware,323,comp.sys.mac.hardware,314.0,41.25%,81.35%
5,5,659,soc.religion.christian,409,talk.religion.misc,151.0,62.06%,84.98%
6,6,358,sci.crypt,345,comp.sys.mac.hardware,1.0,96.37%,96.65%
7,7,84,comp.os.ms-windows.misc,8,comp.sys.mac.hardware,8.0,9.52%,19.05%
8,8,477,rec.sport.hockey,461,0,0.0,96.65%,96.65%
9,9,472,sci.space,403,comp.sys.mac.hardware,1.0,85.38%,85.59%


In [10]:
from openai import OpenAI

items_per_cluster = 10
COMPLETION_MODEL = "gpt-3.5-turbo-instruct"
client = OpenAI(api_key=openai.api_key)

for i in range(num_of_clusters):
    cluster_name = new_df[new_df.cluster == i].iloc[0].rank1
    print(f"Cluster {i}, Rank 1: {cluster_name}, Theme:", end=" ")
    content = "\n".join(
        embedding_df[embedding_df.cluster == i].text.sample(items_per_cluster, random_state=42).values
    )
 
    completions = client.completions.create (
        model=COMPLETION_MODEL,
        prompt=f'''我们想要给下面的内容，分组成有意义的类别，以便我们可以对其进行总结。请根据下面这些内容的共同点，总结一个50个字以内的新闻组的名称。比如 “PC硬件”\n\n内容:\n"""\n{content}\n"""新闻组名称：''',
        max_tokens=100,
        n=1,
        stop=None,
        temperature=0,       
        top_p=1,
    )
    print(completions.choices[0].text.replace("\n", ""))

Cluster 0, Rank 1: rec.autos, Theme: 汽车技术与安全
Cluster 1, Rank 1: comp.sys.ibm.pc.hardware, Theme: 计算机硬件与外设
Cluster 2, Rank 1: talk.politics.misc, Theme: 法律诉讼与税务纠纷
Cluster 3, Rank 1: rec.motorcycles, Theme: Motorcycle Safety and Dog Encounters
Cluster 4, Rank 1: comp.sys.ibm.pc.hardware, Theme: 计算机硬件升级与维护讨论组
Cluster 5, Rank 1: soc.religion.christian, Theme: 宗教信仰这个新闻组名称可以涵盖所有这些内容，因为它们都与宗教信仰有关。宗教信仰是人类文明中最重要的一部分，它影响着人们的价值观、道德观和生活方式。宗教信仰也是人们对
Cluster 6, Rank 1: sci.crypt, Theme: 加密安全与公共安全
Cluster 7, Rank 1: comp.os.ms-windows.misc, Theme: 国际贸易协定这个新闻组主要关注国际贸易协定的签署、谈判、影响和变化。它涵盖了各种国家之间的贸易协定，包括自由贸易协定、双边贸易协定和多边贸易协定
Cluster 8, Rank 1: rec.sport.hockey, Theme: "Hockey News and Analysis"
Cluster 9, Rank 1: sci.space, Theme: 太空探索
Cluster 10, Rank 1: comp.windows.x, Theme: X窗口系统开发与优化
Cluster 11, Rank 1: talk.politics.mideast, Theme: "Global Terrorism and Political Propaganda"
Cluster 12, Rank 1: comp.os.ms-windows.misc, Theme: "Computer Hardware and Software Troubleshooting and Support"
Cluster 13, Ra

In [12]:
items_per_cluster = 1
for i in range(num_of_clusters):
    cluster_name = new_df[new_df.cluster == i].iloc[0].rank1
    print(f"Cluster {i}, Rank 1: {cluster_name}, 抽样翻译:", end=" ")
    content = "\n".join(
        embedding_df[(embedding_df.cluster == i) & (embedding_df.n_tokens < 100)].text.sample(items_per_cluster, random_state=42).values
    )
    completions = client.completions.create (
        model=COMPLETION_MODEL,
        prompt=f'''请把下面的内容翻译成中文\n\n内容:\n"""\n{content}\n"""翻译：''',
        max_tokens=2000,
        n=1,
        stop=None,
        temperature=0,       
        top_p=1,
    )
    
    print(completions.choices[0].text.replace("\n", ""))

Cluster 0, Rank 1: rec.autos, 抽样翻译: 又一个为雅痞设计的吉普车，他们永远不会将其开到越野，但却想要看起来“户外风”。
Cluster 1, Rank 1: comp.sys.ibm.pc.hardware, 抽样翻译: 嗨，这是我给网络的第一条消息（实际上是第三份副本，该死的VI！）。请寻找新的VPIC6.0，它带有更新的VESA 1.2驱动程序，适用于几乎所有已知的显卡。VESA级别为1.2，我的Tseng4000 24位显卡与驱动程序相配合得很好。希望这对你有用！
Cluster 2, Rank 1: talk.politics.misc, 抽样翻译: 艺术
Cluster 3, Rank 1: rec.motorcycles, 抽样翻译: 在马里兰州，我在3年前学习骑行时，每个课程的费用是25美元。现在，初学者骑手课程的费用是125美元，有经验的骑手课程的费用是60美元（诚然，这个课程只需要大约一半的时间）。
Cluster 4, Rank 1: comp.sys.ibm.pc.hardware, 抽样翻译: 我想提一下，索尼不再生产CPD-1304，因为存在几个制造缺陷。新的型号现在是1430，就像苹果的新索尼Trinitrom声称的那样，是14英寸。我不确定缺陷的细节，但我在我们学校的书店工作，可以告诉你，几乎一半的产品都因为某种缺陷被退回。
Cluster 5, Rank 1: soc.religion.christian, 抽样翻译: 如果我没有弄错的话，犹太家族的姓氏科恩、卡恩等被认为是从亚伦后裔的合法指示。家族姓氏利维、莱文等被认为是从利未后裔的合法指示。主要的法律问题是祭司的净化，这涉及到寻找2000年前用于此目的的红母牛的灰烬。
Cluster 6, Rank 1: sci.crypt, 抽样翻译: 在哪里？我真的没有看到任何东西...我不同意，因为已经有其他标准存在。此外，即使他们限制了NREN上的加密，又有什么关系呢？大部分的互联网都是商业性质的。NREN只是为政府和大学研究服务的（读一下提案吧——它是为Cray用户建立的“数据高速公路”，与互联网无关）。
Cluster 7, Rank 1: comp.os.ms-windows.misc, 抽样翻译: 请把下面的内容翻译成中文
Cluster 8, Ra

In [13]:
history = """User : 你是谁？
Assistant : 我是一个人工智能语言模型，可以回答各种问题，包括做菜的问题。
User : 请问鱼香肉丝怎么做？
Assistant : 鱼香肉丝的做法：
1. 猪里脊肉切丝，加入盐、淀粉、料酒搅拌均匀，腌制10分钟
2. 胡萝卜、青椒、木耳切丝备用
3. 热锅凉油，放入蒜末、姜末煸炒出香味，加入肉丝翻炒至变色
4. 加入胡萝卜、青椒、木耳翻炒，加入酱油、糖、醋、盐、味精、水淀粉搅拌均匀，倒入锅中翻炒均匀即可。
提示：鱼香肉丝的关键是调料的配比，酱油、糖、醋、盐、味精的比例要适合自己的口味。
User : 那蚝油牛肉呢？
Assistant : 蚝油牛肉的做法：
1. 将牛肉切成薄片，加入盐、淀粉、料酒腌制10分钟
2. 葱姜切末，蒜切片，青红椒切丝备用
3. 热锅凉油，放入葱姜末、蒜片煸炒出香味，加入牛肉翻炒至变色
4. 加入蚝油、酱油、糖、水淀粉调味，加入青红椒翻炒均匀即可。
提示：蚝油牛肉的关键是牛肉要切薄，翻炒时火候要快，保证牛肉口感鲜嫩。调味时，蚝油和酱油的比例也要适合自己的口味。
"""
def summarize(text, max_tokens=200):
    completions = client.completions.create (
        model=COMPLETION_MODEL,
        prompt=text + "\n\n请总结一下上面User和Assistant聊了些什么：\n",
        max_tokens=max_tokens,
    )
    return completions.choices[0].text
summarized = summarize(history)
print(summarized)


User询问了两道菜的做法，分别是鱼香肉丝和蚝油牛肉。Assistant通过文字介绍了两道菜的详细做法，并给出了一些关键提示。聊天的主题为做菜的方法和调料的搭配技巧。


In [17]:
class Conversation:
    def __init__(self, prompt, num_of_round):
        self.prompt = prompt
        self.num_of_round = num_of_round
        self.messages = []
        self.messages.append({"role": "system", "content": self.prompt})

    def ask(self, question):
        try:
            self.messages.append( {"role": "user", "content": question})
            response = client.chat.completions.create(
                messages=self.messages,
                model="gpt-3.5-turbo",
            )
        except Exception as e:
            print(e)
            return e

        message = response.choices[0].message.content
        self.messages.append({"role": "assistant", "content": message})
        
        if len(self.messages) > self.num_of_round*2 + 1:
            del self.messages[1:3]
        return message

In [16]:
prompt = summarized + "\n\n请你根据已经聊了的内容，继续对话："
conversation = Conversation(prompt, 5)
question = "那宫保鸡丁呢？"
answer = conversation.ask(question)
print("User : %s" % question)
print("Assistant : %s\n" % answer)

User : 那宫保鸡丁呢？
Assistant : 宫保鸡丁的做法也是非常经典的。首先，需要准备鸡胸肉、花生、青椒、红椒和葱姜蒜等食材。接着，将鸡胸肉切成小丁状，用料酒、生抽、淀粉腌制一下。然后将花生干炒一下备用。

下锅热油，爆香葱姜蒜，放入鸡丁翻炒至变色，加入青红椒丁继续翻炒。接着加入适量的郫县豆瓣酱、糖、醋、料酒、生抽、水淀粉搅拌均匀，最后加入炒好的花生即可。

关于宫保鸡丁的关键提示，要注意火候不能太大，保持中小火翻炒，让鸡肉入味且保持嫩滑。此外，豆瓣酱的用量也很重要，要根据个人口味适量添加，不要放过多影响口感。希望你尝试做一下，享受美味的宫保鸡丁！如果有其他菜谱想了解，欢迎继续提问哦！



In [18]:
conversation = Conversation("请你根据已经聊了的内容，继续对话：", 5)
question = "那宫保鸡丁呢？"
answer = conversation.ask(question)
print("User : %s" % question)
print("Assistant : %s\n" % answer)

User : 那宫保鸡丁呢？
Assistant : 噢，宫保鸡丁是一道很经典的中国菜。宫保鸡丁的主要原料是鸡肉丁、花生米、青椒和干辣椒等。烹饪的时候会有一种独特的香味，而且辣辣的口感很受人喜爱。你喜欢吃宫保鸡丁吗？

